— Celda 0 — Instalación

In [1]:
!pip install xgboost==2.0.3 yfinance==0.2.38 pandas numpy --quiet

import xgboost, yfinance, pandas, numpy
print(f"✅ XGBoost:  {xgboost.__version__}")
print(f"✅ yfinance: {yfinance.__version__}")
print(f"✅ Todo listo")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.1/297.1 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 kB 3.3 MB/s eta 0:00:00
✅ XGBoost:  2.0.3
✅ yfinance: 0.2.38
✅ Todo listo


— Celda 1 — Imports y configuración

In [2]:
import os, json, shutil, warnings, logging
from datetime import datetime
import numpy as np
import pandas as pd
import yfinance as yf
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

CONFIG = {
    "DATA_DIR":       "/content/data",
    "MODEL_DIR":      "/content/data/models",
    "SIGNALS_FILE":   "/content/data/signals_output.csv",
    "DRIVE_DIR":      "/content/drive/MyDrive/ETF_Predictor/data",
    "TICKER":         "EXS1.DE",
    "FREQUENCY":      "1wk",

    # Gestión del riesgo
    "STOP_LOSS_ATR_MULT":   1.5,   # Stop-Loss = precio - (ATR × 1.5)
    "TAKE_PROFIT_ATR_MULT": 2.5,   # Take-Profit = precio + (ATR × 2.5)
    "MIN_CONFIDENCE":       0.55,  # Probabilidad mínima para emitir señal
    "RISK_PER_TRADE":       0.02,  # Arriesgar máximo 2% del capital por operación
}

os.makedirs(CONFIG["MODEL_DIR"], exist_ok=True)
print("✅ Configuración cargada")

✅ Configuración cargada


— Celda 2 — Restaurar modelos y archivos desde Drive

In [3]:
# Celda 2 — Restaurar todos los archivos necesarios desde Drive

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

os.makedirs("/content/data/models", exist_ok=True)

DRIVE_DIR   = CONFIG["DRIVE_DIR"]
DRIVE_MODELS = os.path.join(DRIVE_DIR, "models")

# Restaurar CSVs
for filename in ["raw_data.csv", "sentiment_scores.csv", "merged_dataset.csv"]:
    src  = os.path.join(DRIVE_DIR, filename)
    dest = os.path.join(CONFIG["DATA_DIR"], filename)
    if os.path.exists(src):
        shutil.copy2(src, dest)
        print(f"✅ {filename}")
    else:
        print(f"⚠️  No encontrado: {filename}")

# Restaurar modelos
for filename in ["xgb_classifier.json", "xgb_regressor.json",
                 "feature_cols.json", "model_metrics.json"]:
    src  = os.path.join(DRIVE_MODELS, filename)
    dest = os.path.join(CONFIG["MODEL_DIR"], filename)
    if os.path.exists(src):
        shutil.copy2(src, dest)
        print(f"✅ {filename}")
    else:
        print(f"⚠️  No encontrado: {filename}")

print("\n✅ Todos los archivos restaurados")

Mounted at /content/drive
✅ raw_data.csv
✅ sentiment_scores.csv
✅ merged_dataset.csv
✅ xgb_classifier.json
✅ xgb_regressor.json
✅ feature_cols.json
✅ model_metrics.json

✅ Todos los archivos restaurados


Celda 2b — Regenerar merged_dataset.csv desde los dos CSVs disponibles

In [4]:
# Celda 2b — Regenerar merged_dataset.csv desde los dos CSVs disponibles

import pandas as pd
import numpy as np
import os, shutil

def rebuild_merged(data_dir, drive_dir):
    df_raw  = pd.read_csv(f"{data_dir}/raw_data.csv", index_col="date", parse_dates=True)
    df_sent = pd.read_csv(f"{data_dir}/sentiment_scores.csv", index_col="date", parse_dates=True)

    # Alinear índices
    df = df_raw.join(df_sent, how="left", rsuffix="_sent")

    # Forward-fill gaps de sentimiento
    sent_cols = df_sent.columns.tolist()
    df[sent_cols] = df[sent_cols].fillna(method="ffill", limit=2)

    # Eliminar filas sin target
    target_cols = [c for c in df.columns if "target" in c]
    df = df.dropna(subset=target_cols[:1])

    # Guardar localmente y en Drive
    merged_path = f"{data_dir}/merged_dataset.csv"
    df.to_csv(merged_path, index=True, float_format="%.6f")

    dest = os.path.join(drive_dir, "merged_dataset.csv")
    shutil.copy2(merged_path, dest)

    print(f"✅ merged_dataset.csv regenerado: {df.shape}")
    print(f"☁️  Backup en Drive: {dest}")
    return df


df_merged = rebuild_merged(CONFIG["DATA_DIR"], CONFIG["DRIVE_DIR"])

✅ merged_dataset.csv regenerado: (401, 120)
☁️  Backup en Drive: /content/drive/MyDrive/ETF_Predictor/data/merged_dataset.csv


— Celda 3 — Cargar modelos entrenados

In [5]:
# Celda 3 — Cargar modelos corregido

def load_models(model_dir: str):

    with open(os.path.join(model_dir, "feature_cols.json"), "r") as f:
        feature_cols = json.load(f)

    with open(os.path.join(model_dir, "model_metrics.json"), "r") as f:
        metrics = json.load(f)

    # Cargar clasificador usando Booster directamente
    import xgboost as xgb

    clf_booster = xgb.Booster()
    clf_booster.load_model(os.path.join(model_dir, "xgb_classifier.json"))

    reg_booster = xgb.Booster()
    reg_booster.load_model(os.path.join(model_dir, "xgb_regressor.json"))

    logger.info(f"✅ Modelos cargados | Features: {len(feature_cols)}")
    logger.info(f"   Accuracy: {metrics['classifier']['accuracy']*100:.1f}% "
                f"| AUC: {metrics['classifier']['roc_auc']:.3f}")

    return clf_booster, reg_booster, feature_cols, metrics


clf, reg, feature_cols, metrics = load_models(CONFIG["MODEL_DIR"])
print(f"\n✅ Modelos listos | Features requeridos: {len(feature_cols)}")


✅ Modelos listos | Features requeridos: 84
